# Preprocessing Data Teks untuk Analisis Toxicity

Notebook ini berfokus pada tahap preprocessing data teks sebelum dilakukan pelatihan model. Tujuan utama dari proses ini adalah:
- membersihkan data yang tidak relevan atau tidak valid,
- mengurangi noise dan duplikat,
- menstandardisasi bentuk teks agar lebih konsisten,
- menyiapkan fitur teks yang siap dipakai oleh model pada tahap selanjutnya.

Pada bagian berikut, setiap sub-bagian menjelaskan alasan dan hasil dari proses preprocessing yang dilakukan secara berurutan.

In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/indotoxic2024_annotated_data-3.csv")

In [2]:
# hapus spam
df = df[df["is_noise_or_spam_text"] == 0].copy()

# hapus teks yang kosong
empty_mask = (
    df["text"].isna() |
    df["text"].astype(str).str.strip().eq("")
)
df = df[~empty_mask].copy()

### 1. Duplicate Handling

Tujuan bagian ini adalah menghilangkan data yang berulang atau tidak konsisten agar model tidak belajar dari contoh yang sama berulang kali. Proses yang dilakukan meliputi:
- mendeteksi teks duplikat berdasarkan isi teks,
- menyimpan teks yang memiliki label konflik untuk dokumentasi,
- menghapus text yang menimbulkan ambiguitas label,
- menjaga satu representasi terbaik untuk setiap teks unik.

In [3]:
# Mengambil teks duplikat
duplicate_mask = df.duplicated(
    subset=["text"],
    keep=False
)

duplicates = df[duplicate_mask].copy()

# Mengambil teks yang memiliki label konflik
label_conflict = (
    df.groupby("text")["toxicity"]
      .nunique()
      .reset_index(name="n_label")
)

conflict_texts = label_conflict[
    label_conflict["n_label"] > 1
]["text"]

# simpan konflik ke file CSV utk dokumentasi
df_conflict = df[
    df["text"].isin(conflict_texts)
].copy()

df_conflict.to_csv(
    "../data/interim/label_conflict.csv",
    index=False
)

# hapus teks yang memiliki label konflik
df = df[
    ~df["text"].isin(conflict_texts)
].copy()

df = df.drop_duplicates(
    subset=["text"],
    keep="first"
).copy()

In [4]:
print("Shape akhir setelah cleaning:")
print(df.shape)

print("\nDuplicate text:")
print(df["text"].duplicated().sum())

Shape akhir setelah cleaning:
(23009, 17)

Duplicate text:
0


### 2. Ekstraksi Emoji

Bagian ini bertujuan untuk menangkap informasi tambahan dari teks yang sering mengandung ekspresi seperti emoji. Emoji dapat merepresentasikan emosi atau tone komentar, sehingga ekstraksi ini berguna untuk menjaga konteks semantik yang mungkin hilang saat teks dibersihkan. Hasil ekstraksi disimpan di kolom emoji untuk dipelajari lebih lanjut atau dianalisis sebagai fitur pendukung.

In [5]:
import emoji

def extract_emojis(text):
    return " ".join(
        item["emoji"]
        for item in emoji.emoji_list(str(text))
    )

df["emoji"] = df["text"].apply(extract_emojis)

In [6]:
df["emoji"].unique()

array(['', '😔', '👍', ..., '🔥 👍 🤑 💯 👍 🤑 🔥 💯 🤑 🔥 👍 💯', '🏨 🏨 🎀 📩 📩', '🇨🇳 🇲🇾'],
      shape=(2231,), dtype=object)

### 3. Cleaning Text

Tujuan bagian ini adalah membersihkan teks dari karakter noise yang tidak relevan untuk analisis sentimen atau toxicitas. Beberapa proses yang dilakukan meliputi:
- lowercase agar format teks konsisten,
- menghapus emoji, URL, HTML, mention, dan hashtag,
- menghilangkan tanda baca dan karakter berulang,
- menyederhanakan spasi agar teks menjadi lebih rapi dan standar.

Output dari proses ini adalah kolom text_clean yang siap diproses lebih lanjut.

In [7]:
import re

df["text_clean"] = df["text"].copy()

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Hapus emoji
    text = emoji.replace_emoji(text, replace="")
    
    # Hapus URL
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # Hapus HTML tag seperti <br>, <br><br>
    text = re.sub(r"<[^>]+>", " ", text)
    
    # Hapus mention
    text = re.sub(r"@\w+", " ", text)
    
    # Hapus hashtag symbol tetapi pertahankan katanya
    # #SavePalestine -> SavePalestine
    text = re.sub(r"#(\w+)", r"\1", text)
    
    # Hilangkan punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # Normalisasi repeated characters
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    
    # Normalisasi whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text
df["text_clean"] = df["text"].apply(clean_text)


### 4. Slang Normalization

Bagian ini bertujuan untuk menormalkan bahasa informal atau singkatan yang umum dipakai di media sosial. Banyak komentar mengandung kata slang, singkatan, atau variasi ejaan yang berbeda, sehingga perlu dikonversi ke bentuk baku agar model dapat memahami maknanya dengan lebih konsisten. Proses ini meningkatkan kualitas representasi teks sebelum masuk ke tahap pemodelan.

In [8]:
slang_dict = {
    # Negasi
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak",
    "ngga": "tidak", "kaga": "tidak", "kagak": "tidak", "tdk": "tidak",

    # Kata ganti & partikel umum
    "yg": "yang", "dgn": "dengan", "bgt": "banget", "aja": "saja",
    "sy": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "lu": "kamu", "lo": "kamu", "elo": "kamu", "km": "kamu",
    "kmu": "kamu", "elu": "kamu", "w": "saya",

    # Kata tanya & keterangan
    "krn": "karena", "krna": "karena", "karna": "karena",
    "utk": "untuk", "buat": "untuk", "jd": "jadi", "jgn": "jangan",
    "jgnkan": "jangankan", "gmn": "bagaimana", "gimana": "bagaimana",
    "knp": "kenapa", "kenapa": "mengapa", "dmn": "dimana",
    "drmn": "dari mana", "kpn": "kapan", "brp": "berapa",
    "brapa": "berapa", "sm": "sama", "sma": "sama",

    # Waktu
    "skrg": "sekarang", "skrng": "sekarang", "td": "tadi",
    "bsk": "besok", "kmrn": "kemarin", "kmaren": "kemarin",
    "tar": "nanti", "ntar": "nanti", "nti": "nanti",

    # Kata kerja & sifat
    "udh": "sudah", "udah": "sudah", "dah": "sudah", "blm": "belum",
    "blum": "belum", "bkn": "bukan", "bnr": "benar", "bener": "benar",
    "emg": "memang", "emang": "memang", "tau": "tahu", "gatau": "tidak tahu",
    "gtau": "tidak tahu", "pgn": "ingin", "pengen": "ingin",
    "pngen": "ingin", "mau": "ingin", "bs": "bisa", "bsa": "bisa",
    "hrs": "harus", "harusnya": "seharusnya",

    # Kata sifat & ekspresi
    "bgt": "banget", "bngt": "banget", "bener2": "benar-benar",
    "cape": "capai", "capek": "lelah", "cakep": "cantik/tampan", 
    "kece": "keren", "mantul": "mantap betul", "mantep": "mantap",
    "gokil": "gila", "parah": "sangat", "anjay": "ekspresi kaget/kagum",
    "santuy": "santai", "woles": "santai", "baper": "bawa perasaan",
    "kepo": "ingin tahu urusan orang", "julid": "iri/sirik",
    "ambyar": "hancur/berantakan", "receh": "lucu remeh",

    # Singkatan chat umum
    "otw": "on the way / dalam perjalanan", "gpp": "tidak apa-apa",
    "gapapa": "tidak apa-apa", "cmiiw": "correct me if I'm wrong",
    "btw": "ngomong-ngomong", "fyi": "sebagai informasi",
    "pls": "tolong", "thx": "terima kasih", "makasih": "terima kasih",
    "gaje": "tidak jelas", "japri": "jalur pribadi (chat pribadi)",
    "japrii": "jalur pribadi", "wkwk": "tertawa", "wkwkwk": "tertawa",
    "haha": "tertawa", "anjir": "ekspresi kaget", "yaudah": "ya sudah",
    "udahlah": "sudahlah", "kayanya": "kayaknya", "kayaknya": "sepertinya",
    "kyk": "seperti", "kek": "seperti", "sbnrnya": "sebenarnya",
    "sebenarnya": "pada dasarnya", "org": "orang", "org2": "orang-orang",
    "tp": "tapi", "tpi": "tapi", "dr": "dari", "trs": "terus",
    "trus": "terus", "abis": "habis", "abisin": "habiskan",
    "duit": "uang", "cuan": "keuntungan/uang", "gaje": "tidak jelas",
}

In [9]:
def normalize_repeated_char(text):
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    return text

def normalize_slang(text):
    words = text.split()
    
    normalized = [
        slang_dict.get(word, word)
        for word in words
    ]
    
    return " ".join(normalized)

df["text_clean"] = df["text_clean"].apply(normalize_repeated_char)
df["text_clean"] = df["text_clean"].apply(normalize_slang)

### 5. Topic Handling

Tujuan dari bagian ini adalah menyiapkan struktur data untuk kolom topic agar lebih mudah diproses dan divisualisasikan. Teks topic biasanya berbentuk string yang berisi beberapa kategori dengan pemisah koma, sehingga perlu diubah menjadi list agar lebih efektif untuk analisis lebih lanjut, seperti pemetaan tema atau eksplorasi label.

In [10]:
df["topic_list"] = (
    df["topic"]
    .fillna("")
    .apply(
        lambda x: [
            topic.strip()
            for topic in str(x).split(",")
            if topic.strip()
        ]
    )
)

In [11]:
df[["topic", "topic_list"]].head()

,topic,topic_list
0,Disabilitas,[Disabilitas]
1,Jewish,[Jewish]
2,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"
3,"Terpolarisasi, Jewish","[Terpolarisasi, Jewish]"
6,"Terpolarisasi, Tionghoa","[Terpolarisasi, Tionghoa]"


### 6. Cek Hasil Akhir

Bagian ini berfungsi sebagai validasi kualitas preprocessing. Tujuannya adalah memastikan tidak ada teks kosong yang tersisa setelah proses pembersihan dan bahwa data yang sudah dibersihkan siap digunakan pada tahapan berikutnya. Proses pengecekan ini penting agar hasil akhir konsisten dan meminimalkan error dalam pelatihan model.

In [12]:
empty_after_cleaning = (
    df["text_clean"].str.strip().eq("")
)

print(
    "Text kosong setelah cleaning:",
    empty_after_cleaning.sum()
)
df = df[~empty_after_cleaning].copy()

Text kosong setelah cleaning: 0


In [14]:
pd.set_option("display.max_colwidth", 200)

df[
    ["text", "emoji", "text_clean", "topic_list"]
].sample(10)

,text,emoji,text_clean,topic_list
4511,gila ini koleksi jersey edisi 100 tahun Persis bagus semua 💯,💯,gila ini koleksi jersey edisi 100 tahun persis bagus semua,[Disabilitas]
31121,"Seganteng ini aja diselingkuhi, apalagi gue yg dibandingin dengan sepatu nya szobo masih cakepan sepatu szobo. Njir lah emang wanita seperti itu kah? Yg ganteng aja dibikin galau setengah gila sam...",,seganteng ini saja diselingkuhi apalagi saya yang dibandingin dengan sepatu nya szobo masih cakepan sepatu szobo njir lah memang wanita seperti itu kah yang ganteng saja dibikin galau setengah gil...,[Disabilitas]
41274,"Hajar Bawahannya Sebelum Rapat, Menhan Tuai Sorotan! . . Umur 22 Mata Najwa Shihab Adakami LGBT CCTV JISOO Piting Merangkul Rempang FISIP UI Selasa Sore Mendung Pegawai Nagita Slavina Bahlul Sudok...",,hajar bawahannya sebelum rapat menhan tuai sorotan umur 22 mata najwa shihab adakami lgbt cctv jisoo piting merangkul rempang fisip ui selasa sore mendung pegawai nagita slavina bahlul sudoku zhao...,"[LGBTQ+, Terpolarisasi]"
5765,"231016 China News Weibo Updated China News Network memproduksi lagu penghormatan ""Kawan Seperjuangan"" untuk peringatan 10 tahun inisiatif ""One Belt, One Road"". Lagu ""Kawan Seperjuangan"" yang dinya...",,231016 china news weibo updated china news network memproduksi lagu penghormatan kawan seperjuangan untuk peringatan 10 tahun inisiatif one belt one road lagu kawan seperjuangan yang dinyanyikan o...,[Tionghoa]
9206,ASIAN GAMES 2022 HANGZHOU buletin sports FINAL SPEED RELAY PUTRA PANJAT TEBING INDONESIA CHINA FALSE START 🥈 MEDALI PERAK!\n\nTim Indonesia harus puas dengan medali perak pada nomor speed relay pu...,🥈 🥇 🥈 🥉,asian games 2022 hangzhou buletin sports final speed relay putra panjat tebing indonesia china false start medali perak tim indonesia harus puas dengan medali perak pada nomor speed relay putra ka...,[Tionghoa]
8210,"Presiden AS Siap Kirim Dukungan Usai Israel Diserang Hamas, Perang Besar Palestina?",,presiden as siap kirim dukungan usai israel diserang hamas perang besar palestina,[Jewish]
2603,"Perang Israel vs Hamas, Ini Kata Para Ekonom Soal Pasar Global",,perang israel vs hamas ini kata para ekonom soal pasar global,[Jewish]
15050,"Potret Indahnya Kebhinekaan\nHADIRI HUT PSMTI, \nGANJAR DISAMBUT HANGAT RIBUAN RAKYAT ETNIS TIONGHOA!\n\nCapres 2024, Ganjar Pranowo disambut antusias oleh ribuan pengurus Tionghoa se Indonesia ya...",,potret indahnya kebhinekaan hadiri hut psmti ganjar disambut hangat ribuan rakyat etnis tionghoa capres 2024 ganjar pranowo disambut antusias oleh ribuan pengurus tionghoa se indonesia yang tergab...,"[Terpolarisasi, Tionghoa]"
6486,"MURID MADRASAH DUKUNG PRESIDEN ANIES [RETWEET UNTUK MENANGKAN RP.165 RIBU CENTANG BIRU] Hamas Kiai Thoifur 13,4 rb Berita · Israel 2,37 jt Zionis #AniesMuhaimin2024 65,3 rb Gibran 28,9 rb JANGAN D...",,murid madrasah dukung presiden anies retweet untuk menangkan rp 165 ribu centang biru hamas kiai thoifur 13 4 rb berita israel 2 37 jt zionis aniesmuhaimin2024 65 3 rb gibran 28 9 rb jangan deh 6 ...,[Jewish]
4836,"Selamat Pagi JAHANAM! Iya kamu! ini buat kamu ISRAEL yg maksa warga gaza ngungsi biar selamat, tp pas mereka ngungsi, kalian BOM! B I N A T A N G K A L I A N‼️🔥",‼️ 🔥,selamat pagi jahanam iya kamu ini untuk kamu israel yang maksa warga gaza ngungsi biar selamat tapi pas mereka ngungsi kalian bom b i n a t a n g k a l i a n,"[Terpolarisasi, Jewish]"
